# CoT injection demo — visualize prefill + regeneration

Takes one cached chain of thought from `data/qa_inference_small.parquet`, corrupts it
with **random numerical tokens** (a control-style injection) at **cutoff_frac = 0.5**
using `build_injection_prefill`, and regenerates the rest with `rollout_cot`.

Shows: the original CoT → the truncated-and-injected **prefill** (injected numbers
highlighted) → the **regenerated** continuation. The injection splices one randomly
sampled number token (`is_number`, blocklist applied) at every sentence boundary in
the first half of the trace, as exact token ids. See `SCOTSPRINT.md` §4.

In [1]:
# --- CPU thread caps for the cgroup-throttled H100 pod (load .env before importing torch)
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

import os
import torch

torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", torch.get_num_threads())))

In [2]:
import html
import random
import re
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

from subliminality import (
    get_device,
    token_mask,
    is_number,
    build_injection_prefill,
    build_reasoning_prompt,
    build_boxed_answer_instruction,
    rollout_cot,
    answer_scaffold_ids,
    answer_candidate_token,
    batched_answer_scores,
    DEFAULT_THINK_BUDGET,
)

device = get_device()
SEED = 0
print("device:", device)

device: cuda


## Load the model

In [3]:
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map=device, dtype=torch.bfloat16).eval()
END_THINK_ID = tokenizer.convert_tokens_to_ids("</think>")
print("loaded; </think> id =", END_THINK_ID)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded; </think> id = 128014


## Pick one cached chain of thought

We take the longest **naturally closed** trace (a complete, coherent CoT) so the 50%
cutoff has several sentences to inject into. The question text isn't stored in the
inference output, so we join back to the source dataset on `qid`.

In [4]:
def data_path(name):
    return next(p for p in [Path("data") / name, Path("../data") / name] if p.exists())

cache = pd.read_parquet(data_path("qa_inference_small.parquet"))
src = pd.read_parquet(data_path("wm-non-ambiguous-hard-2.parquet"))
df = cache.merge(src[["qid", "q_str_open_ended", "x_name", "y_name"]], on="qid")

closed = df[~df.forced_close]
row = (closed if len(closed) else df).sort_values("n_think_tokens", ascending=False).iloc[0]
print("qid:", row.qid, "| prop:", row.prop_id, "| think tokens:", row.n_think_tokens, "| forced_close:", row.forced_close)
print("Q:", row.q_str_open_ended[:200])
print("correct:", row.correct_name, "| incorrect:", row.incorrect_name)

qid: 96e708ae6a3d47db6f7e0af459fa300ef5b0009d1b96b75e8aac89e986db95ff | prop: wm-us-college-long | think tokens: 992 | forced_close: False
Q: Which place is located more to the west: George Washington University School of Nursing, DC or Middlesex University, MA?
correct: George Washington University School of Nursing, DC | incorrect: Middlesex University, MA


## The original chain of thought

In [5]:
think_ids = list(row.think_token_ids)
original_cot = tokenizer.decode(think_ids, skip_special_tokens=False)
print(original_cot)

Okay, so I need to figure out which place is more to the west: George Washington University School of Nursing, DC or Middlesex University, MA. Hmm, let me think about this step by step.

First, I should probably clarify what "west" means in terms of geography. Generally, in the US, west refers to the western part of the country, so anything towards the left when facing north. But since both universities are located in the eastern part of the US, I guess I need to look at their specific regions.

George Washington University is in Washington, D.C., which is the capital of the United States. I know Washington, D.C., is located on the Potomac River, between Maryland and Virginia. So, it's in the mid-Atlantic region. Now, considering the direction, if I'm facing north, west would be towards the left. But since both universities are on the East Coast, west would mean heading towards the southern parts of the US, but I think that's not the case here. Maybe I'm overcomplicating it.

Wait, per

## Build a random number-token pool

`token_mask(tok, is_number)` flags every vocab token that decodes to a plain run of
digits not in `NUMBER_BLOCKLIST`; we sample 10 of them (seeded) as the control pool
the injection draws from.

In [6]:
number_ids = torch.nonzero(token_mask(tokenizer, is_number), as_tuple=False).squeeze(-1).tolist()
pool = random.Random(SEED).sample(number_ids, 10)
print(f"{len(number_ids)} number tokens in vocab; sampled pool:")
print([tokenizer.decode([i]) for i in pool])

1059 number tokens in vocab; sampled pool:
['594', '677', '001', '379', '948', '739', '924', '436', '738', '493']


## Corrupt at cutoff_frac = 0.5

`build_injection_prefill` splits the CoT into sentences, keeps the first ~50% (snapped
to a sentence boundary), and splices one sampled number token in at **each** sentence
boundary in that prefix. The kept text is the model's exact original tokens; each
injected number is an exact id. Injected numbers are highlighted below.

In [7]:
CUTOFF = 0.5
pf = build_injection_prefill(think_ids, insert_ids=pool, cutoff_frac=CUTOFF, seed=SEED,
                             tokenizer=tokenizer, end_think_id=END_THINK_ID)
print(f"{pf.n_sentences} sentence boundaries | cut at token {pf.cutoff_token}/{len(think_ids) - 1} "
      f"| injected {pf.n_injected} numbers")

def show_cot(prefix_text, gen_text=None):
    "Render a CoT: highlight injected (NUM) in yellow; the regenerated tail in blue."
    esc = re.sub(r"\(\s*(\d+)\s*\)",
                 r'<span style="background:#ffe08a;font-weight:bold">(\1)</span>',
                 html.escape(prefix_text))
    body = esc + (f'<span style="background:#cfe8ff">{html.escape(gen_text)}</span>' if gen_text else "")
    display(HTML(f'<div style="white-space:pre-wrap;font-family:monospace;font-size:12px;'
                 f'line-height:1.45">{body}</div>'))

show_cot(tokenizer.decode(pf.prefill_ids, skip_special_tokens=False))

52 sentence boundaries | cut at token 499/991 | injected 28 numbers


## Regenerate the rest of the CoT

Feed the injected prefill back through `build_reasoning_prompt(..., prefill=...)` and
`rollout_cot`. Below: the kept-and-injected prefix (injections in **yellow**) followed
by the freshly **regenerated** continuation (in **blue**), through the new `</think>`.

In [8]:
instruction = build_boxed_answer_instruction(row.q_str_open_ended, [row.x_name, row.y_name])
prompt = build_reasoning_prompt(tokenizer, instruction, prefill=pf.prefill_ids)
rollout = rollout_cot(model, tokenizer, [prompt], end_think_id=END_THINK_ID,
                      max_new_tokens=DEFAULT_THINK_BUDGET, seed=SEED)[0]
print(f"regenerated {len(rollout.think_ids)} tokens after the prefill | forced_close: {rollout.forced_close}")

show_cot(tokenizer.decode(pf.prefill_ids, skip_special_tokens=False),
         tokenizer.decode(rollout.think_ids, skip_special_tokens=False))

regenerated 444 tokens after the prefill | forced_close: False


## Bonus — did the answer move?

Read `logprob_diff = logP(correct) − logP(incorrect)` at the `\boxed{` scaffold for the
regenerated trace, vs. the cached baseline (these random numbers are a *control*, so we
don't expect a systematic push — the real test compares entangled vs. control).

In [9]:
answer_ids = answer_scaffold_ids(tokenizer)
cand = [(answer_candidate_token(tokenizer, row.correct_name),
         answer_candidate_token(tokenizer, row.incorrect_name))]
after = batched_answer_scores(model, [rollout.full_ids], cand,
                              pad_id=tokenizer.eos_token_id, answer_ids=answer_ids)[0]
print(f"baseline logprob_diff (cor-inc): {row.logprob_diff:+.3f}")
print(f"after random injection:          {after.logprob_diff:+.3f}")

baseline logprob_diff (cor-inc): +6.875
after random injection:          -9.250
